<div style="font-size: 0.85em; line-height: 1.5;">

<h3>Parent Document Retriever</h3>

<p><strong>What is it?</strong><br>
A retriever that stores small chunks for search but returns larger parent chunks when answering. This gives the LLM more context.</p>

<p><strong>How it works</strong></p>
<ol>
  <li>Split documents into large parent chunks.</li>
  <li>Further split each parent into small child chunks.</li>
  <li>Embed and store child chunks in the vector store.</li>
  <li>When querying, retrieve child chunks, then return the parent chunk(s).</li>
  <li>Pass the parent chunk(s) as context to the LLM.</li>
</ol>

<p><strong>Why use it?</strong></p>
<ul>
  <li><strong>Better context</strong> – small chunks for search, large chunks for answer.</li>
  <li><strong>Improves recall</strong> – captures full answer even if only part matched.</li>
  <li><strong>Great for long documents</strong> – PDFs, articles, manuals.</li>
</ul>

<p><strong>Visualisation</strong></p>
<pre>
Indexing:
[Large parent chunks] → split into small child chunks → embed child chunks

Retrieval:
Query → find similar child chunk(s)
         ↓
Return corresponding parent chunk(s)
         ↓
LLM uses parent chunk(s) as context
</pre>

<p><strong>Implementation</strong><br>
We will use LangChain’s <code>ParentDocumentRetriever</code> with a vector store and a document splitter.</p>

</div>

In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader, TextLoader
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryByteStore

from dotenv import load_dotenv

 Load Relevant Documents

In [2]:
health_pdf = PyPDFLoader('../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf').load()
crop_pdf = PyPDFLoader

In [3]:
files = [
    ('../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf', 'pdf'),
    ('../../04_data_ingestion_document_processing/data/crop_disease.pdf', 'pdf'),
    ('../../04_data_ingestion_document_processing/data/agriculture.html', 'html'),
    ('../../04_data_ingestion_document_processing/data/agriculture.txt', 'txt'),
]

all_docs = []

for path, file_type in files:
    if file_type == 'pdf':
        loader = PyPDFLoader(path)        
    elif file_type == 'html':
        loader = BSHTMLLoader(path, open_encoding='utf-8', bs_kwargs={'features': 'html.parser'})  
    elif file_type == 'txt':
        loader = TextLoader(path, encoding='utf-8')       
    else:
        continue
    all_docs.extend(loader.load())
    
print(f'Loaded {len(all_docs)} documents.') 
    

Loaded 37 documents.


Create Parent and Child Splitters

In [4]:
# Parent splitter: creates larger chunks (for context)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,  # larger parent chunks
    chunk_overlap = 100
)

# Child splitter: creates smaller chunks (for retrieval)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 150,      # smaller child chunks
    chunk_overlap = 30
)
print('Parent and child splitters ready')

Parent and child splitters ready


* **Parent splitter** will create large chunks (600 characters) to give the LLM more context.

* **Child splitter** will create small chunks (150 characters) for precise retrieval.



Create Vector Store and ParentDocumentRetriever


In [5]:
# Embedding model
embeddings = OpenAIEmbeddings()

# Create In-memory byte store for parent documents
byte_store = InMemoryByteStore()

# Create an in‑memory Chroma vector store.
# It will store the small child chunks and their embeddings.
vectorstore = Chroma(
    collection_name='parent_doc_demo',
    embedding_function=embeddings,
    persist_directory=None
)

# Instantiate the ParentDocumentRetriever.
# This combines the vector store (child chunks) and byte store (parent chunks)
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    byte_store=byte_store,
    docstore=None,    # will use an in-memory store by default
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

# Add documents (splits into parents and children, stores children)
parent_retriever.add_documents(all_docs, ids=None)

print('ParentDocumentRetriever ready.')


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


ParentDocumentRetriever ready.


Test ParentDocumentRetriever with a Broad Question

In [6]:
# Define a broad question
query = 'What are the major diseases affecting crops and livestock in Nigeria, and how are they managed?'

# Retrieve documents using the parent retriever
docs = parent_retriever.invoke(query)

print(f'Retrieved {len(docs)} parent chunk(s)\n')

for i, doc in enumerate(docs, start=1):
    print(f'Chunk {i} (length={len(doc.page_content)} characters):')
    print(doc.page_content)
    print('-' * 100)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved 4 parent chunk(s)

Chunk 1 (length=421 characters):
Agriculture in Nigeria: Crops, Livestock, and Disease Management



Agriculture in Nigeria
Last updated: August 2026


Introduction

            Agriculture is one of the most important sectors of the Nigerian economy. It contributes significantly to the country's GDP and employs more than a third of the labour force. In rural areas, agriculture is the primary source of income and food security.
        


Major Crops
----------------------------------------------------------------------------------------------------
Chunk 2 (length=541 characters):
3. Mastitis
   Affected animal: Dairy cattle and goats
   Symptoms: Swollen udder, abnormal milk, fever.
   Control: Good milking hygiene and antibiotic treatment under veterinary guidance.

Challenges Facing Agriculture in Nigeria

- Poor rural infrastructure such as roads and storage facilities.
- Limited access to credit and modern equipment.
- Climate change causing drought a

In [7]:
# More focused question about crop diseases only
query = 'What are the common crop diseases and their control methods?'

# Retrieve parent chunks
docs = parent_retriever.invoke(query)

print(f'Retrieved {len(docs)} parent chunk(s):\n')
for i, doc in enumerate(docs, start=1):
    print(f'Chunk {i} (length={len(doc.page_content)} characters):')
    print(doc.page_content)
    print('-' * 60)

Retrieved 4 parent chunk(s):

Chunk 1 (length=507 characters):
Major Crops

Cassava – a major staple crop.
Yam – widely grown in the middle belt.
Maize – important for food and livestock feed.
Rice – local production is growing.
Sorghum and Millet – common in the north.
Beans – key source of protein.



Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infected plants early.
------------------------------------------------------------
Chunk 2 (length=437 characters):
Common Crop Diseases and Control

1. Cassava Mosaic Disease
   Affected crop: Cassava
   Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
   Control: Use disease-free cuttings, plant resistant varieties, and remove infected plants.

2. Maize Smut
   Affected crop: Maize
   Symptoms: Grey or black galls on ea